# 如何缓存 LLM 响应

LangChain 为 LLM 提供了一个可选的 [缓存](/docs/concepts/chat_models/#caching) 层。这在两个方面非常有用：

*   **节省成本**：如果您经常多次请求相同的完成，它可以通过减少您向 LLM 提供商发出的 API 调用次数来为您节省资金。
*   **提高速度**：它可以加快您的应用程序速度，方法是减少您向 LLM 提供商发出的 API 调用次数。

In [ ]:
%pip install -qU langchain_openai langchain_community

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass()
# Please manually enter OpenAI Key

In [2]:
from langchain_core.globals import set_llm_cache
from langchain_openai import OpenAI

# To make the caching really obvious, let's use a slower and older model.
# Caching supports newer chat models as well.
llm = OpenAI(model="gpt-3.5-turbo-instruct", n=2, best_of=2)

In [3]:
%%time
from langchain_core.caches import InMemoryCache

set_llm_cache(InMemoryCache())

# The first time, it is not yet in cache, so it should take longer
llm.invoke("Tell me a joke")

CPU times: user 546 ms, sys: 379 ms, total: 925 ms
Wall time: 1.11 s


"\nWhy don't scientists trust atoms?\n\nBecause they make up everything!"

In [4]:
%%time
# The second time it is, so it goes faster
llm.invoke("Tell me a joke")

CPU times: user 192 µs, sys: 77 µs, total: 269 µs
Wall time: 270 µs


"\nWhy don't scientists trust atoms?\n\nBecause they make up everything!"

## SQLite 缓存

SQLite uses a **page cache** to store recently accessed database pages in memory. This improves performance by reducing the number of disk I/O operations.

SQLite 使用**页面缓存**将最近访问的数据库页面存储在内存中。这通过减少磁盘 I/O 操作次数来提高性能。

### Cache Modes

SQLite supports several cache modes, which can be configured using the `PRAGMA cache_size` statement.

SQLite 支持多种缓存模式，可以通过 `PRAGMA cache_size` 语句进行配置。

*   **`DEFAULT`**: The default cache mode. The cache size is determined by the `cache_size` PRAGMA.
    *   **`DEFAULT`**: 默认缓存模式。缓存大小由 `cache_size` PRAGMA 决定。
*   **`NORMAL`**: A normal cache mode. The cache size is determined by the `cache_size` PRAGMA.
    *   **`NORMAL`**: 普通缓存模式。缓存大小由 `cache_size` PRAGMA 决定。
*   **`BACKUP`**: A cache mode optimized for backups. It uses a smaller cache to minimize memory usage during backup operations.
    *   **`BACKUP`**: 针对备份优化的缓存模式。它使用较小的缓存以在备份操作期间最小化内存使用。

### Cache Size

The `cache_size` PRAGMA controls the number of pages that SQLite will store in the cache. A larger cache size can improve performance for read-heavy workloads, but it also increases memory usage.

`cache_size` PRAGMA 控制 SQLite 将在缓存中存储的页面数量。较大的缓存大小可以提高读密集型工作负载的性能，但也会增加内存使用量。

The `cache_size` can be set to a positive integer, which represents the number of pages. If `cache_size` is set to a negative integer, SQLite will use a default cache size based on available memory.

`cache_size` 可以设置为一个正整数，表示页面数量。如果 `cache_size` 设置为负整数，SQLite 将根据可用内存使用默认缓存大小。

```sql
-- Set the cache size to 2000 pages
PRAGMA cache_size = 2000;

-- Use the default cache size
PRAGMA cache_size = -1;
```

```sql
-- 将缓存大小设置为 2000 页
PRAGMA cache_size = 2000;

-- 使用默认缓存大小
PRAGMA cache_size = -1;
```

### Cache Flushing

SQLite automatically flushes the cache when necessary, for example, when the cache is full or when a write operation occurs. You can also manually flush the cache using the `PRAGMA cache_flush` statement.

SQLite 在必要时会自动刷新缓存，例如，当缓存已满或发生写入操作时。您也可以使用 `PRAGMA cache_flush` 语句手动刷新缓存。

```sql
PRAGMA cache_flush;
```

```sql
PRAGMA cache_flush;
```

### Cache Tuning

The optimal cache size depends on your specific workload and available memory. You may need to experiment with different `cache_size` values to find the best performance.

最佳缓存大小取决于您的具体工作负载和可用内存。您可能需要尝试不同的 `cache_size` 值来找到最佳性能。

For read-heavy workloads, a larger cache size can significantly improve performance. For write-heavy workloads, a smaller cache size might be more appropriate to reduce memory contention.

对于读密集型工作负载，较大的缓存大小可以显著提高性能。对于写密集型工作负载，较小的缓存大小可能更合适，以减少内存争用。

Consider the following when tuning the cache:

在调整缓存时，请考虑以下几点：

*   **Available Memory**: Ensure that the cache size does not exceed the available memory.
    *   **可用内存**: 确保缓存大小不超过可用内存。
*   **Workload Characteristics**: Analyze your workload to determine whether it is read-heavy or write-heavy.
    *   **工作负载特性**: 分析您的工作负载，以确定它是读密集型还是写密集型。
*   **Performance Monitoring**: Monitor your application's performance to identify any bottlenecks related to caching.
    *   **性能监控**: 监控您应用程序的性能，以识别与缓存相关的任何瓶颈。

In [5]:
!rm .langchain.db

In [6]:
# We can do the same thing with a SQLite cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path=".langchain.db"))

In [7]:
%%time
# The first time, it is not yet in cache, so it should take longer
llm.invoke("Tell me a joke")

CPU times: user 10.6 ms, sys: 4.21 ms, total: 14.8 ms
Wall time: 851 ms


"\n\nWhy don't scientists trust atoms?\n\nBecause they make up everything!"

In [8]:
%%time
# The second time it is, so it goes faster
llm.invoke("Tell me a joke")

CPU times: user 59.7 ms, sys: 63.6 ms, total: 123 ms
Wall time: 134 ms


"\n\nWhy don't scientists trust atoms?\n\nBecause they make up everything!"